# ATiG 2026: LT-FH exercise using ltpred

## What you will learn

The [ltpred tutorial](https://bvilhjal.github.io/ltpred/tutorial/) builds a
simulated register whose genetic liability is known, then scores every person
from family history, and ends by listing shortcuts it never measured. In this
exercise you measure two of them on the same register: what a wrong
liability-scale heritability does to the score (Part A), and how far a
posterior mean genetic liability is from a disease risk (Part B). Part C
estimates the heritability from the register itself with a tetrachoric
correlation, and Part D moves an observed-scale GWAS h² — and the effective
sample sizes behind it — onto the liability scale.

Everything is compared against the true genetic liability the simulation drew;
no real data are used. Budget one hour for Parts A and B; Parts C and D are homework.

Before running each part, write down what you expect; afterwards, explain one
checkpoint number in your own words. Keep code and answers in this notebook.
Your final product is the Q13 note distinguishing what the score shows from
what it cannot establish.

The model is LT-FH++ ([Pedersen et al. 2022](https://doi.org/10.1016/j.ajhg.2022.01.009)),
which extends LT-FH ([Hujoel et al. 2020](https://doi.org/10.1038/s41588-020-0613-6))
with age of onset. ADuLT ([Pedersen et al. 2023](https://doi.org/10.1038/s41467-023-41210-z))
is the same personalised construction without family history. The scores in this
exercise are computed with the deterministic Pearson–Aitken engine of PA-FGRS
([Dybdahl Krebs et al. 2024](https://doi.org/10.1016/j.ajhg.2024.09.009)). Across
five psychiatric disorders, PA-FGRS and polygenic scores were weakly correlated yet
complementary ([Dybdahl Krebs et al. 2026](https://doi.org/10.1016/j.ajhg.2025.11.016)),
as two noisy estimates of one additive genetic liability should be; Part B's
ceiling shows the same logic from the other side. ltpred's
[algorithm page](https://bvilhjal.github.io/ltpred/algorithm/) states the model and its
[connection to selection-index prediction](https://bvilhjal.github.io/ltpred/algorithm/#connection-to-selection-index-and-blup).


## Setup

Python 3.9 or newer with NumPy and SciPy, and ltpred — not yet on PyPI, so
install it from source. The commands are the same on Windows, macOS and Linux;
no compiler and no R are needed:

```bash
git clone https://github.com/bvilhjal/ltpred.git
cd ltpred
pip install -e ".[fast]"      # plain  pip install -e .  works too; numba only speeds it up
```

No data files either: Part 0 rebuilds the tutorial register from fixed seeds,
so a fresh session is enough — you do not need the tutorial still open. Parts
A and B run in about a minute; Part C factorises a 10,000-person kinship
matrix and needs a few GB of memory. Executed with ltpred 0.7.0 (commit `2d85ce9`); the printouts
are identical under NumPy 1.26, 2.2 and 2.4. Run the cell below to confirm
your install.


In [1]:
import numpy as np
from scipy.stats import norm, rankdata

import ltpred
print("ltpred", ltpred.__version__)

ltpred 0.7.0


## Part 0. Rebuild the tutorial cohort

These cells are tutorial steps 1, 3 and 4 with the tutorial's seeds, so the names
below (`cohort`, `scores`, `est`, `predicted`, `pred_est`, `at_risk`) are the
tutorial's and the checkpoints match its printouts. If your tutorial session is
still open, run them anyway: Part B needs `predicted.var`.

In [2]:
from ltpred import simulate_pedigree, simulate_register_liabilities

H2 = 0.5                                      # liability-scale heritability
CIP_K, CIP_MID, CIP_SLOPE = 0.10, 60.0, 1.0 / 8.0
EVAL_AGE = 70.0                               # everyone is followed to here
INDEX_AGE = 40.0                              # the prospective cut in step 4

# the generating cumulative-incidence curve, on a 1-year age grid
AGE_GRID = np.arange(0, 121, 1.0)
TRUE_CIP = CIP_K / (1.0 + np.exp((CIP_MID - AGE_GRID) * CIP_SLOPE))

ids, father, mother = simulate_pedigree(
    np.random.default_rng(20260921), n_founder_pairs=100, gens=2)

cohort = simulate_register_liabilities(
    np.random.default_rng(20260921), ids, father, mother,
    h2=H2, cip_ages=AGE_GRID, cip_values=TRUE_CIP, eval_age=EVAL_AGE)

print(f"{len(ids)} people, {int(cohort.status.sum())} diagnosed by age "
      f"{EVAL_AGE:.0f} ({cohort.status.mean():.1%})")
print(f"var(true genetic liability) = {cohort.genetic.var():.4f}  (target {H2})")

984 people, 81 diagnosed by age 70 (8.2%)
var(true genetic liability) = 0.5083  (target 0.5)


**Checkpoint.** 984 people, 81 diagnosed, variance 0.5083: tutorial step 1.
`cohort.genetic` is the truth that everything below is compared against.

In [3]:
from ltpred import estimate_liabilities

scores = estimate_liabilities(
    cohort.ids, cohort.father, cohort.mother,
    probands=cohort.ids,
    status=cohort.status.astype(int), age=cohort.age,
    use="gwas",                          # the proband's own diagnosis is used
    cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K,
    h2=H2)

est = np.asarray(scores.est)
print(f"corr(score, true genetic liability) = {np.corrcoef(est, cohort.genetic)[0, 1]:.4f}")
print(f"score sd = {est.std():.4f}   mean posterior variance = {np.asarray(scores.var).mean():.4f}")

corr(score, true genetic liability) = 0.5502
score sd = 0.3898   mean posterior variance = 0.3558


In [4]:
at_risk = cohort.onset > INDEX_AGE          # still undiagnosed at the cut
index_time = cohort.birth_time + INDEX_AGE
probands_at_risk = [p for p, keep in zip(cohort.ids, at_risk) if keep]

predicted = estimate_liabilities(
    cohort.ids, cohort.father, cohort.mother,
    probands=probands_at_risk,
    status=cohort.status.astype(int), age=cohort.age,
    use="prediction",                    # the proband's own diagnosis is hidden
    cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K, h2=H2,
    birth_time=cohort.birth_time, index_time=index_time[at_risk])

pred_est = np.asarray(predicted.est)
print(f"{int(at_risk.sum())} of {len(ids)} probands are disease-free at age {INDEX_AGE:.0f}")
print(f"corr(score, truth) = {np.corrcoef(pred_est, cohort.genetic[at_risk])[0, 1]:.4f}"
      f"   score sd = {pred_est.std():.4f}")

974 of 984 probands are disease-free at age 40
corr(score, truth) = 0.3069   score sd = 0.2134


**Checkpoint.** Correlations 0.5502 and 0.3069: tutorial steps 3 and 4. Step 3
also noted that 0.390² + 0.356 = 0.508, the variance of `cohort.genetic`: the
variance of the scores plus the mean posterior variance recovers Var(g). Part A
asks what becomes of that identity when h² is wrong.

## Part A. What does a wrong h² do?

Step 3 passed the true `h2=0.5`. In a real analysis h² is an external estimate for
the disease on the liability scale, and it has no default: the
[method guide](https://bvilhjal.github.io/ltpred/guide/) says to prefer an external estimate or to cross-check it.
Here you can pass a wrong value on purpose and compare with the truth.

### Q1: If you tell the scorer h² = 0.8 when the truth is 0.5, do the scores spread more or less? Does their ranking change?

**Before running:** write your expectation down with one sentence of reasoning.
The scorer's model says that a fraction h² of the liability variance is genetic
and shared with relatives.

### Q2: Score the register under h² = 0.2, 0.5 and 0.8 and tabulate the consequences.

For each assumed value, score with `use="gwas"` exactly as in Part 0 and print
five numbers: the correlation with `cohort.genetic`; the variance of the scores;
the mean posterior variance `s.var`; their sum; and the slope of `cohort.genetic`
regressed on the score, `np.polyfit(est, cohort.genetic, 1)[0]`.

In [ ]:
print(f"{'assumed h2':>10} {'corr':>7} {'var(score)':>11} {'mean var':>9} {'sum':>7} {'slope':>7}")
for h2_assumed in (0.2, 0.5, 0.8):
    s = estimate_liabilities(
        cohort.ids, cohort.father, cohort.mother, probands=cohort.ids,
        status=cohort.status.astype(int), age=cohort.age, use="gwas",
        cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K,
        h2=...)                                # <- the assumed value
    e, v = np.asarray(s.est), np.asarray(s.var)
    corr = ...                                 # correlation with cohort.genetic
    slope = ...                                # np.polyfit(e, cohort.genetic, 1)[0]
    print(f"{h2_assumed:>10.1f} {corr:>7.4f} {e.var():>11.4f} {v.mean():>9.4f} "
          f"{e.var() + v.mean():>7.4f} {slope:>7.3f}")

**Checkpoint.** The `sum` column should come out close to the h² you passed in,
whichever it was, and the row for 0.5 reproduces Part 0.

### Q3: Which columns moved and which did not? What does a slope of 1 mean, and why does only the right h² give it?

A posterior mean is *calibrated* when the truth regressed on it has slope 1: among
probands scored 0.3, the true liability averages 0.3. Relate your slopes to the
guide's advice on h². Optional: scatter `cohort.genetic` against the score for
each h² and draw the fitted line.

## Part B. Who becomes a case between 40 and 70?

Step 4's answer says that its correlation of 0.31 "is not the accuracy of
predicting who becomes a case after 40". Measure that accuracy. The 974 probands in
`predicted` were disease-free at 40; an *incident* case is one of them diagnosed by
70, which the simulation knows through `cohort.onset`.

### Q4: How many of the 974 probands become incident cases, and what is the incidence?

Hint: `cohort.status` means "diagnosed by 70" and `at_risk` means "undiagnosed at
40". Combine them, then index with `at_risk` so that the result lines up with
`pred_est`.

In [ ]:
y = ...          # boolean, one per proband in `predicted`: diagnosed after 40 and by 70
print(f"{int(y.sum())} incident cases among {len(y)} probands ({y.mean():.1%})")

### Q5: Compute the AUC for three predictors of that outcome on the same 974 people.

The AUC is the probability that a random incident case scores above a random
non-case. With ranks from `scipy.stats.rankdata` it is the Mann–Whitney statistic:
if `r` are the ranks of the predictor, `n1` the number of cases and `n0` the number
of non-cases, AUC = (sum of the cases' ranks − n1(n1 + 1)/2) / (n1 · n0).

The predictors: the prospective score `pred_est`; the true genetic liability
`cohort.genetic[at_risk]`, which is the ceiling for any predictor built from
genetics alone; and the step-3 score `est` restricted to the same people, which
used each proband's own diagnosis.

**Before running:** order the three AUCs.

In [ ]:
def auc(x, y):
    r = rankdata(x)
    n1, n0 = y.sum(), (~y).sum()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

print(f"AUC prospective score         = {auc(pred_est, y):.3f}")
print(f"AUC true genetic liability    = {auc(..., y):.3f}")
print(f"AUC step-3 score, same people = {auc(..., y):.3f}")

**Checkpoint.** The step-3 score should sit near 1. If it does not, you have not
restricted it to the probands in `predicted`.

### Q6: Why is the ceiling below 1 although it is the truth? Where does the gap between the prospective score and the ceiling come from? What does the third AUC say about `use="gwas"` in a prospective claim?

Use step 3's decomposition: of Var(g) = 0.5, the relatives and the proband's own
diagnosis identified about 0.15, and about 0.36 remained posterior uncertainty.

### Q7: Turn the score into a risk and check it against the observed incidence.

A posterior mean liability is not a probability, but with its posterior variance
it becomes one. Under the model, a proband's full liability given the relatives is
Normal with mean `m = pred_est` and variance `s² = predicted.var + 1 − h²`: the
residual is independent of the relatives because `use="prediction"` hid the
proband's own status. With thresholds `T_a = Φ⁻¹(1 − CIP(a))`, being disease-free
at 40 means `l ≤ T40` and being diagnosed by 70 means `l > T70`, so

risk = [Φ((T40 − m)/s) − Φ((T70 − m)/s)] / Φ((T40 − m)/s).

Compute it for every proband. Compare the mean risk with the incidence from Q4.
Then split the probands into three equal-size groups by rank of `pred_est` and
compare observed incidence with mean risk in each group.

In [ ]:
m, v = pred_est, np.asarray(predicted.var)
T40 = norm.ppf(1.0 - np.interp(INDEX_AGE, AGE_GRID, TRUE_CIP))
T70 = norm.ppf(1.0 - np.interp(EVAL_AGE, AGE_GRID, TRUE_CIP))
s = np.sqrt(v + 1.0 - H2)
p_free_at_40 = norm.cdf((T40 - m) / s)
risk = ...                                    # the formula above

print(f"mean model risk {risk.mean():.4f}   observed incidence {y.mean():.4f}")

thirds = np.array_split(np.argsort(m, kind="stable"), 3)
for label, g in zip(("low", "middle", "high"), thirds):
    print(f"{label:>7}  n={len(g)}  observed {y[g].mean():.3f}  model risk {risk[g].mean():.3f}")

**Checkpoint.** The mean model risk should agree with the observed incidence to
within a few tenths of a percent. Why does the risk need `predicted.var` and not
only the score? With 71 events, how many groups can this cohort support?

## Part C. Get h² from the register itself (homework)

Part A showed that h² is not a nuisance parameter. The [method guide](https://bvilhjal.github.io/ltpred/guide/)
suggests a cross-check: h² is about twice a first-degree tetrachoric correlation,
and `ltpred.tetrachoric` estimates one from the 2×2 table of two binary statuses.

### Q8: Estimate h² from parent–offspring pairs in the tutorial register, then in a larger one.

Pair every person with each parent whose id is in the table (both parents give a
pair) and call `tetrachoric(parent_status, child_status)`. Report 2ρ ± 2 SE. Then
simulate a register with `n_founder_pairs=1000` and seed 1 for both generators and
repeat. Stay at or below 1000 founder pairs on a laptop: the simulator factorises
the full kinship matrix.

In [ ]:
from ltpred.tetrachoric import tetrachoric


def parent_offspring_status(reg):
    idx = {p: i for i, p in enumerate(reg.ids)}
    parent, child = [], []
    for kid, fa, mo in zip(reg.ids, reg.father, reg.mother):
        for par in (fa, mo):
            if par in idx:

                parent.append(...); child.append(...)   # statuses of the pair
    return np.asarray(parent), np.asarray(child)


par, kid = parent_offspring_status(cohort)
t = tetrachoric(par, kid)
print(f"{len(par)} pairs, {int((par & kid).sum())} both diagnosed: "
      f"rho = {t.rho:+.3f} +/- {t.se:.3f}  ->  h2 = {2 * t.rho:+.2f} +/- {2 * t.se:.2f}")

big_ids, big_fa, big_mo = simulate_pedigree(
    np.random.default_rng(1), n_founder_pairs=1000, gens=2)
big = simulate_register_liabilities(
    np.random.default_rng(1), big_ids, big_fa, big_mo,
    h2=H2, cip_ages=AGE_GRID, cip_values=TRUE_CIP, eval_age=EVAL_AGE)
...

**Checkpoint.** Both estimates should bracket 0.5, the second with about a third
of the first's standard error.

### Q9: Why does twice the tetrachoric work here, and what would break it in a real register?

Consider what the tetrachoric assumes about the threshold, what this simulation
assumes about shared environment, and how the register was sampled.

## Part D. Scales and effective sample sizes (homework)

A real analysis rarely starts on the liability scale: GREML or LDSC on a
case/control GWAS reports h² on the observed 0/1 scale, and oversampling cases
makes that scale depend on the sample's case proportion P. The bridge is the
transformation of [Lee et al. 2011](https://doi.org/10.1016/j.ajhg.2011.02.002)
(extended in [Lee et al. 2012](https://doi.org/10.1002/gepi.21614)): with
prevalence K and z = φ(Φ⁻¹(1 − K)),

h²_liab = h²_obs · K²(1 − K)² / (z² · P(1 − P)),

where the P factor applies only to designs that oversample cases. ltpred
ships it as `observed_to_liability_h2` and its inverse
`liability_to_observed_h2`. The same P(1 − P) sets the study's effective
sample size Neff = 4 N₁ N₀ / N: information about a binary outcome scales
with N · P(1 − P), which is why GWAS oversample cases and report Neff, not N.

### Q10: What observed-scale h² should each design expect here?

The truth is h² = 0.5 with K = 0.10. Use `liability_to_observed_h2` to compute
the h² each design would report: (i) the register as sampled (81 cases among
984, so P = 81/984), (ii) a balanced case/control study of the same 984
people, and (iii) a population sample, where the P factor does not apply.
Convert each back with `observed_to_liability_h2` and check that you recover
0.5.

In [ ]:
from ltpred import observed_to_liability_h2, liability_to_observed_h2

designs = {"register as sampled": ...,   # P = 81/984
           "balanced case/control": ...,
           "population sample": ...}     # no oversampling: prop_cases=None
for name, P in designs.items():
    obs = ...                           # liability_to_observed_h2(H2, CIP_K, P)
    back = ...                          # observed_to_liability_h2(obs, CIP_K, P)
    print(f"{name:20s} P={...:.3f}  h2_obs={...:.4f}  round trip={...:.4f}")

**Checkpoint.** 0.14, 0.48 and 0.17 all round-trip to 0.5 exactly. The three
designs measure the same disease, yet their observed-scale h² differ by more
than a factor of three.

### Q11: The register's GWAS reports h² = 0.14. Feed it to the scorer unconverted — what happens?

Score the register exactly as in Part A, but at the observed-scale value from
Q10 (i) instead of a liability-scale h². Report the correlation with
`cohort.genetic` and the calibration slope. Which Part A row does this
imitate, and why is no error raised? Optional: convert the balanced design's
h² from Q10 while forgetting the P factor, and try to score at the result.

In [ ]:
obs_register = ...                       # liability_to_observed_h2(H2, CIP_K, 81/984)
s = estimate_liabilities(
    cohort.ids, cohort.father, cohort.mother, probands=cohort.ids,
    status=cohort.status.astype(int), age=cohort.age, use="gwas",
    cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K,
    h2=...)                              # the unconverted observed-scale value
e = np.asarray(s.est)
print(f"corr = {...:.4f}  slope = {...:.3f}")

**Checkpoint.** Correlation ≈ 0.55 but slope ≈ 2.5 — the over-shrunk regime
of Part A's h² = 0.2 row, only worse. The forgotten P factor turns 0.48 into
1.39, which the scorer rejects outright: h² must be in (0, 1].

### Q12: How much information does each design carry?

Compute Neff = 4 N₁ N₀ / N for the register as sampled, the same 984 people
balanced 492/492, a biobank GWAS with 5,000 cases and 45,000 controls, and a
consortium with 50,000 cases and 150,000 controls. Express Neff/N through P
alone. Why do GWAS oversample cases to roughly 1:1 to 1:4 instead of
genotyping population samples, and why does the Lee transformation need P
while Part C's tetrachoric correlation did not?

In [ ]:
for name, n_case, n_ctrl in [("register as sampled", ..., ...),
                             ("balanced", 492, 492),
                             ("biobank", 5_000, 45_000),
                             ("consortium", 50_000, 150_000)]:
    n = n_case + n_ctrl
    neff = ...                           # 4 * n_case * n_ctrl / n
    print(f"{name:20s} N={n:6d}  P={n_case / n:.3f}  "
          f"Neff={neff:7.0f}  Neff/N={neff / n:.3f}")

**Checkpoint.** 297, 984, 18,000 and 150,000, with Neff/N = 4P(1 − P). The
tetrachoric inferred its threshold from the pairs' own marginal rates — a
population sample — while an observed-scale h² is defined on the sample's 0/1
scale and must be told how that sample was drawn.

### Q13: Analysis note.

In at most 150 words, state what the family-history score estimates, what it
showed in this exercise, and what it cannot establish. Cite one number from each
of Parts A and B and, if you did them, Parts C and D.

---
Part of [ATIG 2026](https://github.com/bvilhjal/ATIG_2026), teaching day
[24 September](https://github.com/bvilhjal/ATIG_2026/tree/main/teaching_days/2026-09-24).